In [16]:
%load_ext autoreload
%autoreload 2

"""
AI News Curator Agent using LangGraph with Agentic Browser Tool
The agent uses browser automation to intelligently scrape news.smol.ai
"""
from scripts.linkedin_agent_utils import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Database Logging Setup

Initialize the SQLite database logger to track all agent runs with timestamps.

In [17]:
from scripts.db_logger import AgentRunLogger

# Initialize the database logger
db_logger = AgentRunLogger(db_path="linkedin_agent_runs.db")
print("✅ Database logger initialized")

2026-02-11 08:39:32,363 - scripts.db_logger - INFO - Database initialized at linkedin_agent_runs.db


✅ Database logger initialized


In [18]:

llm = ChatOpenAI(
    base_url="http://localhost:1234/v1/",
    # base_url="http://localhost:11434/v1/",
    api_key="lm-studio",  # LM Studio doesn't require a real API key,
    model="qwen3:8b",
    # model="gpt-oss:20b",
    temperature=0.5,
    reasoning_effort="none",
    streaming=False,
)

In [19]:
llm.invoke("hello /nothink")  # Warm up the model

AIMessage(content='<think>\n\n</think>\n\nHello! How can I assist you today? 😊', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 12, 'total_tokens': 27, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'qwen-3@q4_1', 'system_fingerprint': 'qwen-3@q4_1', 'id': 'chatcmpl-g0pixye04b966nmxauve8n', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--291f4141-8cf2-4a76-9bd4-3576a152bf98-0', usage_metadata={'input_tokens': 12, 'output_tokens': 15, 'total_tokens': 27, 'input_token_details': {}, 'output_token_details': {}})

In [20]:
"""Run the agent with database logging"""
print("🚀 Starting Agentic AI News Curator...\n")


# Initialize state
initial_state = {
    "raw_headlines": [],
    "texts": [],
    "top_stories": [],
    "research_data": {},
    "linkedin_post": "",
    "search_query": "",
    "llm": llm,
    "url": "https://news.smol.ai/issues"
}

# Create the agent
app = create_agent()

try:
    # Run the agent
    final_state = await app.ainvoke(initial_state)
    
    # Log successful run to database
    run_id = db_logger.log_run(final_state, status="success")
    print(f"\n✅ Run logged to database with ID: {run_id}")
    
except Exception as e:
    # Log failed run to database
    print(f"\n❌ Error during agent run: {e}")
    run_id = db_logger.log_run(
        initial_state, 
        status="error", 
        error_message=str(e)
    )
    print(f"Error logged to database with ID: {run_id}")
    raise

🚀 Starting Agentic AI News Curator...

State of the dict: {'url': 'https://news.smol.ai/issues', 'llm': ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x137095130>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x13711e540>, root_client=<openai.OpenAI object at 0x137097ad0>, root_async_client=<openai.AsyncOpenAI object at 0x1370a7f20>, model_name='qwen3:8b', temperature=0.5, model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='http://localhost:1234/v1/', reasoning_effort='none'), 'texts': [], 'raw_headlines': [], 'top_stories': [], 'search_query': '', 'linkedin_post': ''}
{'url': 'https://news.smol.ai/issues', 'llm': ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x137095130>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x13711e540>, root_client=<openai.OpenAI object at 0x137097ad0>, root_async_client=<o

2026-02-11 08:39:59,949 - scripts.linkedin_agent_utils - INFO - Selected story: News(news_title='Recursive Language Models (RLMs) Introduce Novel Approach', description='Recursive Language Models (RLMs) introduce a novel approach using a second programmatic context space to extend long-context capabilities.', date='Feb 09')
2026-02-11 08:40:00,807 - scripts.linkedin_agent_utils - INFO - Generated search query: Recursive Language Models long-context capabilities
2026-02-11 08:40:01,609 - scripts.linkedin_agent_utils - INFO - ✍️ Generating LinkedIn post...
2026-02-11 08:40:17,256 - scripts.linkedin_agent_utils - WARNING - Structured output failed, trying manual parsing: 1 validation error for LinkedInPost
  Invalid JSON: expected value at line 1 column 1 [type=json_invalid, input_value='<think>\n\n</think>\n\nR...Models #MachineLearning', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/json_invalid
2026-02-11 08:40:27,568 - scripts.linkedin_agent_util


✅ Run logged to database with ID: 96


In [21]:
final_state["retrieved_data"].summary

'Recursive Language Models (RLMs) address long‑context limitations by treating the context as an external, programmatically manipulable resource rather than forcing the base model to ingest ever‑larger windows; the root model generates code that recursively decomposes a massive dataset into smaller chunks, invokes child agents or sub‑models on each chunk, and then aggregates their structured outputs, so the primary model never sees the full corpus at once. This approach goes beyond traditional Retrieval‑Augmented Generation, which can lose critical evidence if retrieval fails, by giving the agent full control over its own context through an interactive coding environment (a REPL) that can split, query, and recombine information on demand. As a result, RLMs enable scalable, deep reasoning over effectively unbounded data without retraining or the massive computational cost of expanding the native context window, making them a practical solution for enterprise‑scale tasks that require coh

In [ ]:
if isinstance(final_state["linkedin_post"], LinkedInPost):
    print("✅ LinkedIn post generated successfully!")
    combined_with_url = f"{final_state['linkedin_post'].post}" # \n\nRead more: {final_state['linkedin_post'].url}"
    print(f"Post Content with URL:\n{combined_with_url}\n")


✅ LinkedIn post generated successfully!
Post Content with URL:
Xiaomi's MiMo-V2-Flash is making waves in the AI world with its impressive balance of power and efficiency. This model, packed with 309 billion parameters, manages to keep only 15 billion active during inference—like a superhero using just the right amount of energy for a mission. It can handle massive contexts up to 256,000 tokens while churning out responses at 150 tokens per second. That's fast enough to keep up with your thoughts as you brainstorm in real time.

The real magic happens with its hybrid attention and multi-tensor parallel setup. Think of it as a well-organized team where each member knows their role, allowing the model to scale smoothly without breaking a sweat. And the cost? Just $0.1 per million input tokens and $0.3 for output—way cheaper than hiring a full-time AI expert.

What really sets MiMo-V2-Flash apart is its lightweight design. Unlike some models that need a whole lab of hardware, this one runs

In [ ]:
final_state["retrieved_data"].links_w_contents
# print(final_state['linkedin_post'].post)

# print(type(final_state["linkedin_post"]))
# isinstance(final_state["linkedin_post"], LinkedInPost)
# from IPython.display import Image, display

# display(Image(app.get_graph().draw_mermaid_png()))

[('https://videocardz.com/newz/nvidia-launches-nemotron-3-nano-30b-open-weight-models-with-1m-token-context-length',
  'Nemotron 3 Nano is described as a hybrid Mamba-Transformer mixture-of-experts model. NVIDIA lists it as a 30B total and 3B active parameter model, while the'),
 ('https://research.nvidia.com/labs/nemotron/files/NVIDIA-Nemotron-3-White-Paper.pdf',
  'Following) 2-Bench (T ool Use) SWE-Bench (Coding) LCB v6 (Coding) RULER @ 1M (Long Ctx) ISL/OSL 8k/16k 0 20 40 60 80 100 Accuracy (%) 67.7 89.1 99.2 71.5 49.0 38.8 68.2 86.3 57.8 85.0 51.0 47.7 22.0 66.0 77.5 48.5 91.7 98.7 65.0 47.5 34.0 61.0 N/A +tools: Accuracy Throughput Nemotron-3-Nano-30B-A3B Qwen3-30B-A3B-Thinking-2507 GPT-OSS-20B-A4B 0 1 2 3 4 5 6 7 8 Relative Throughput (Output tokens/s/GPU) 3.3 1.0 1.5 Figure 2 | The hybrid Mamba-Transformer MoE architecture used by Nemotron 3 models can achieve state-of-the-art accuracy on leading reasoning benchmarks and ultra-long-context tasks while providing throughput impro

## Query Database

View recent agent runs and their results from the database.

In [9]:
# View recent runs
import pandas as pd

recent_runs = db_logger.get_recent_runs(limit=10)
df = pd.DataFrame(recent_runs)
print(f"Recent {len(df)} runs:")
df

Recent 10 runs:


,id,timestamp,status,linkedin_post,linkedin_post_url,search_query,iteration_count,error_message
0,24,2025-10-31 09:03:03,success,Looped LLMs are shaking up the world of large ...,https://www.arxiv.org/pdf/2510.24824,Looped LLMs efficiency gains comparison,0,None
1,23,2025-10-31 07:26:59,success,"So, MiniMax M2 has taken a step back in its de...",https://www.rohan-paul.com/p/minimax-m2-model-...,MiniMax M2 multi-hop reasoning full attention,0,None
2,22,2025-10-31 07:21:13,success,Imagine if your favorite AI model could do mor...,https://www.arxiv.org/pdf/2510.24824,Looped LLMs efficiency gains comparison,0,None
3,21,2025-10-31 06:53:03,success,Kimi Linear isn't just another model—it's a ga...,https://x.com/SonglinYang4,Kimi Linear KDA long-context processing effici...,0,None
4,20,2025-10-31 05:38:28,error,None,None,,0,'list' object has no attribute 'artifact'
5,19,2025-10-31 05:18:08,success,The world of AI coding just got a whole lot mo...,https://news.smol.ai/issues/25-10-28-openai-re...,"AI coding agents advancements: Cursor 2.0, Com...",0,None
6,18,2025-10-31 05:13:47,success,\n\nThe world of AI coding agents is heating u...,https://news.smol.ai/issues/25-10-28-openai-re...,"AI coding agents advancements: Composer-1, Cur...",0,None
7,17,2025-10-31 05:09:10,error,None,None,,0,'News' object has no attribute 'descriptio'
8,16,2025-10-31 05:04:09,error,None,None,,0,1 validation error for extract_important_info\...
9,15,2025-10-31 05:00:33,error,None,None,,0,1 validation error for extract_important_info\...


In [10]:
# View details of a specific run
run_details = db_logger.get_run_details(run_id)

print(f"📊 Run Details for ID {run_id}:")
print(f"Timestamp: {run_details['timestamp']}")
print(f"Status: {run_details['status']}")
print(f"Iterations: {run_details['iteration_count']}")
print(f"\nArticles scraped: {len(run_details['articles'])}")
print(f"Retrieved sources: {len(run_details['retrieved_sources'])}")

if run_details['linkedin_post']:
    print(f"\n📝 LinkedIn Post:\n{run_details['linkedin_post'][:200]}...")
    
# View articles
if run_details['articles']:
    articles_df = pd.DataFrame(run_details['articles'])
    print("\n📰 Articles:")
    display(articles_df[['article_title', 'article_date', 'is_selected']])

📊 Run Details for ID 24:
Timestamp: 2025-10-31 09:03:03
Status: success
Iterations: 0

Articles scraped: 5
Retrieved sources: 5

📝 LinkedIn Post:
Looped LLMs are shaking up the world of large language models by offering a smarter way to build and use these powerful tools. Instead of stacking hundreds of layers like a tower of blocks, looped LLM...

📰 Articles:


,article_title,article_date,is_selected
0,Moonshot AI Releases Kimi Linear (KDA) with Da...,Oct 30,1
1,MiniMax M2 Pivots to Full Attention for Multi-...,Oct 30,1
2,"ByteDance, Princeton, and Mila Introduce Loope...",Oct 30,1
3,OpenAI's Aardvark (GPT-5) Enters Private Beta ...,Oct 30,1
4,Cursor Launches Faster Cloud Coding Agents wit...,Oct 30,1


In [ ]:
# View all successful LinkedIn posts
successful_posts = db_logger.get_successful_posts(limit=20)
posts_df = pd.DataFrame(successful_posts)

print(f"✅ {len(posts_df)} Successful Posts Generated:")
posts_df

In [ ]:
# Export specific run to JSON for backup or analysis
db_logger.export_to_json(f"agent_run_{run_id}.json", run_id=run_id)
print(f"✅ Exported run {run_id} to JSON")

In [35]:
for x in final_state["top_stories"].artifact:
    for y in (x[1]):
        print(y)

news_title='LangChain & LangGraph 1.0 Released' description='Major updates for reliable, controllable agents and unified documentation.' date='Oct 22'
news_title='vLLM Release Notes' description='Not much happened today' date='Oct 22'
news_title="ChatGPT Atlas: OpenAI's AI Browser" description='' date='Oct 21'
news_title='gemini atlas openai google langchain ivp capitalg' description='' date=''
news_title='DeepSeek-OCR: A New Vision-Language Model for Efficient Text Decoding' description='The article discusses the release of DeepSeek-OCR, a novel vision-language model that compresses long text as visual context with high accuracy and efficiency. It challenges traditional tokenization approaches by achieving ~97% decoding precision at <10× compression and processing up to ~33M pages/day on 20 A100-40G nodes, outperforming benchmarks like GOT-OCR2.0.' date='Oct 20'


{'r': 3, 'c': 5}


In [6]:
from langgraph.graph import StateGraph

# No specific schema - accepts any dict
graph = StateGraph(dict)
# ... build graph ...
app = graph.compile()

# This works - any keys are allowed
result = app.invoke({
    "my_data": {"key": "value"},
    "any_other_key": "anything"
})

ValueError: Graph must have an entrypoint: add at least one edge from START to another node

In [27]:
import numpy as np

def anomaly_score(current, baseline, beta=0.5, c=10):
    abs_diff = abs(current - baseline)
    denom = baseline ** beta + c
    score = abs_diff / denom
    return score

def is_anomaly(current, baseline, threshold=3.):
    score = anomaly_score(current, baseline)
    return score > threshold

# Testing
print(is_anomaly(12, 10))    # False, score ~0.15
print(is_anomaly(100, 50))   # True or False depends on threshold (score ~2.93)
print(is_anomaly(1200, 1000))
print(is_anomaly(500, 80))

False
False
True
True
